# 07 — Satış ve Dolum Analizi

`transactions.csv` ve `deliveries.csv` — pompa decimal, test satışı, hayali/algılanmayan dolum sinyalleri.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name != 'eda':
    ROOT = ROOT / 'eda'
sys.path.insert(0, str(ROOT))

from utils.data_loader import load_all
from utils.plots import setup_style

setup_style()
dfs = load_all()
tx = dfs['transactions']
deliv = dfs['deliveries']

In [ ]:
# Satış istatistikleri
print('Toplam işlem:', len(tx))
print('Test satışı:', tx['test_satisi'].sum())
print('Satış tipi:', tx['satis_tipi'].value_counts().to_dict())
print('Litre istatistikleri:')
print(tx['litre'].describe())

In [ ]:
# Pompa decimal adayı: litre vs tutar/birim_fiyat
tx_ok = tx.dropna(subset=['litre','tutar','birim_fiyat'])
tx_ok = tx_ok[tx_ok['birim_fiyat'] > 0]
tx_ok['litre_hesap'] = tx_ok['tutar'] / tx_ok['birim_fiyat']
tx_ok['litre_oran'] = tx_ok['litre'] / tx_ok['litre_hesap']

# 10x sapma
decimal = tx_ok[(tx_ok['litre_oran'] > 5) | (tx_ok['litre_oran'] < 0.2)]
print('Decimal şüpheli işlem:', len(decimal))
if len(decimal):
    display(decimal.groupby(['istasyon_kodu','tank_no']).size().sort_values(ascending=False).head())

In [ ]:
# İşlem büyüklüğü dağılımı
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tx['litre'].hist(bins=80, ax=axes[0], edgecolor='k', alpha=0.7)
axes[0].set_xlabel('Litre'); axes[0].set_title('Tekil satış litre dağılımı')
tx.groupby('istasyon_kodu')['litre'].sum().plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='k')
axes[1].set_title('Toplam satış — istasyon')
plt.tight_layout()
plt.show()

In [ ]:
# Dolum analizi
print('Toplam dolum kaydı:', len(deliv))
print('Dolum net istatistikleri:')
print(deliv['dolum_net'].describe())

deliv['tarih'] = deliv['dolum_baslangic'].dt.normalize()
dolum_gun = deliv.groupby(['istasyon_kodu','tank_no','tarih'])['dolum_net'].sum().reset_index()
print('\nEn büyük dolumlar:')
display(dolum_gun.nlargest(10, 'dolum_net'))

In [ ]:
# Dolum sonrası envanter artışı vs kayıtlı dolum — ue1t join
ue1t = dfs['ue1t_30min']
dolum_donem = ue1t[ue1t['tanka_dolum'] > 100][['saat_1','istasyon_kodu','tank_no','tanka_dolum','kayip_kazanc']]
print('UE1T dolumlu dönem (>100 lt):', len(dolum_donem))
display(dolum_donem.nlargest(10, 'tanka_dolum'))

Sonraki: manifold ve bölmeli tanklar (`08_manifold_bolmeli.ipynb`).